# Лекция и методическое руководство: Проектирование и модификация SQL-таблиц
## Конспект преподавателя (для объяснения у доски и живой демонстрации)

> **Как пользоваться этим блокнотом на паре:**
> * Блоки **«📌 На доску»** содержат емкие синтаксические конструкции и правила для переноса на доску.
> * Блоки **«💡 Зачем это нужно»** дают понятные аналогии для объяснения физического смысла студентам.
> * Блоки **«⚙️ Как это работает»** раскрывают поведение СУБД.
> * Блоки **«💻 Демонстрация»** содержат готовые минимальные примеры кода для запуска на проекторе.
> * Блоки **«⚠️ Грабли новичков»** перечисляют ошибки, которые студенты гарантированно сделают, если их не предупредить.

---

### План занятия (тайминг на 90 минут):
1. **DDL: Скелет таблицы и типы данных** (10 мин)
2. **Система ограничений целостности (Constraints)** (15 мин)
3. **Внешние ключи (FOREIGN KEY) и команда `PRAGMA`** (10 мин)
4. **Парадигма DDL First и миграция данных из CSV** (15 мин)
5. **Параметризованный `INSERT` и перехват `IntegrityError`** (10 мин)
6. **Безопасный `UPDATE` и золотое правило `WHERE`** (10 мин)
7. **Удаление `DELETE`, запрет сирот и `ON DELETE CASCADE`** (10 мин)
8. **Транзакции: принцип неделимости (`COMMIT` / `ROLLBACK`)** (10 мин)


In [1]:
import sqlite3
import pandas as pd

# Создадим чистую базу в памяти для интерактивных демонстраций у доски
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

print("✅ Демонстрационная СУБД в памяти готова к лекции.")


✅ Демонстрационная СУБД в памяти готова к лекции.


---
## Тема 1. DDL: Создание таблицы и типы данных

### 💡 Зачем это нужно:
До этого момента мы работали с готовыми таблицами в Pandas или просто выгружали DataFrame в базу через `df.to_sql()`. При автоматической выгрузке SQLite сам угадывает типы колонок (и часто ошибается). 
В продакшене и реальных проектах базу проектирует человек: мы жестко задаем архитектуру, типы данных и правила допустимости.

### 📌 На доску:
```sql
CREATE TABLE имя_таблицы (
    имя_колонки_1 ТИП_ДАННЫХ ОГРАНИЧЕНИЕ,
    имя_колонки_2 ТИП_ДАННЫХ ОГРАНИЧЕНИЕ
);
```

**Базовые типы данных SQLite:**
* `INTEGER` — целые числа (id, количество, счетчики);
* `REAL` — числа с плавающей точкой (цены, баланс, средние баллы);
* `TEXT` — любые строки (имена, email, даты в ISO);
* `BLOB` — бинарные файлы (фото, документы — на практике в БД хранят пути к ним).


---
## Тема 2. Система ограничений целостности (Constraints)

### 💡 Зачем это нужно:
База данных — это «охранник на входе». Если пользователь ввел отрицательный тариф, пустой пароль или дубликат почты, база должна **отклонить запись на уровне СУБД**, а не надеяться на аккуратность программиста.

### 📌 На доску (Таблица ограничений):
| Ограничение | Что делает | Пример на доску |
|---|---|---|
| `PRIMARY KEY` | Первичный ключ. В SQLite `INTEGER PRIMARY KEY` = автоинкремент (1, 2, 3...) | `pc_id INTEGER PRIMARY KEY` |
| `NOT NULL` | Запрет пустого значения (`NULL` недопустим) | `nickname TEXT NOT NULL` |
| `UNIQUE` | Запрет дубликатов (создает уникальный B-Tree индекс) | `email TEXT UNIQUE` |
| `CHECK (...)` | Проверка бизнес-логики через выражение | `CHECK (hourly_rate > 0)` |
| `DEFAULT val` | Значение по умолчанию, если поле не передано | `DEFAULT 'Active'` |

### 💻 Демонстрация у доски: создание таблицы компьютеров


In [ ]:
cur.execute("""
CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL CHECK (zone_name IN ('Standard', 'Bootcamp', 'VIP', 'PS5')),
    hourly_rate REAL NOT NULL CHECK (hourly_rate > 0),
    status      TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Maintenance'))
);
""")
conn.commit()
print("Таблица hardware_pcs успешно создана со всеми ограничениями.")


Таблица hardware_pcs успешно создана со всеми ограничениями.


---
## Тема 3. Внешние ключи (FOREIGN KEY) и связывание таблиц

### 💡 Зачем это нужно:
Внешний ключ запрещает появление «сирот» — записей в дочерней таблице, которые ссылаются на несуществующего родителя.
*Пример:* нельзя открыть игровую сессию на клиента с `id = 9999`, если в клубе зарегистрировано всего 50 человек.

### 📌 На доску:
```sql
CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id  INTEGER NOT NULL,
    pc_id      INTEGER NOT NULL,
    -- Определение внешних ключей в конце таблицы:
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (pc_id) REFERENCES hardware_pcs(pc_id)
);
```

### ⚠️ Главная засада SQLite (записать крупно на доске!):
> **В SQLite проверка внешних ключей ПО УМОЛЧАНИЮ ОТКЛЮЧЕНА!**  
> При каждом новом соединении с базой нужно обязательно первой строчкой выполнить:
> `conn.execute("PRAGMA foreign_keys = ON;")`  
> Без этого `FOREIGN KEY` будет декорацией — SQLite разрешит вставить любой мусор.


In [3]:
# Создадим таблицы клиентов и сессий
cur.executescript("""
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL UNIQUE,
    balance       REAL NOT NULL DEFAULT 0 CHECK (balance >= 0)
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER NOT NULL,
    pc_id          INTEGER NOT NULL,
    hours_played   REAL NOT NULL CHECK (hours_played > 0),
    status         TEXT NOT NULL DEFAULT 'active' CHECK (status IN ('active', 'completed')),
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (pc_id) REFERENCES hardware_pcs(pc_id)
);
""")
conn.commit()
print("Схема 1:N (members -> game_sessions <- hardware_pcs) создана.")


Схема 1:N (members -> game_sessions <- hardware_pcs) создана.


---
## Тема 4. Парадигма DDL First и правильный импорт CSV в SQLite

### 💡 Зачем это нужно:
В Лабе 4 мы делали `df.to_sql('table', conn, if_exists='replace')`.
**Почему в реальных проектах это табу?**
Режим `replace` удаляет таблицу и генерирует заново «плоскую» болванку:
* Уничтожаются `PRIMARY KEY` и автоинкремент;
* Стираются связи `FOREIGN KEY`;
* Удаляются все проверки `CHECK` и правила `UNIQUE`.

### 📌 На доску:
```python
# ❌ АНТИПАТТЕРН (сносит всю схему DDL и ограничения):
df.to_sql("members", conn, if_exists="replace")

# ✅ ИНЖЕНЕРНЫЙ ПОДХОД (DDL First):
# 1. Сначала создаем схему через DDL (CREATE TABLE со всеми CHECK и FK)
# 2. Затем добавляем данные:
df.to_sql("members", conn, if_exists="append", index=False)
```

### ⚙️ Порядок импорта имеет решающее значение:
Так как включен `PRAGMA foreign_keys = ON;`, базу нельзя заполнять в произвольном порядке:
1. **Сначала** загружаются родительские таблицы-справочники (`members`, `hardware_pcs`).
2. **Только затем** — дочерняя таблица операций (`game_sessions`).

### 💻 Демонстрация краша: что будет, если нарушить порядок загрузки


In [4]:
# Сымитируем данные из CSV в виде DataFrame
df_sessions_sample = pd.DataFrame([
    {"session_id": 1, "member_id": 10, "pc_id": 1, "hours_played": 2.0, "status": "active"}
])

print("Попытка загрузить сессию для клиента member_id = 10, которого еще нет в members:")
try:
    df_sessions_sample.to_sql("game_sessions", conn, if_exists="append", index=False)
except Exception as e:
    print(f"❌ ОШИБКА СУБД: {e}")
    print("👉 Вывод: дочернюю таблицу нельзя импортировать раньше родительской!")


Попытка загрузить сессию для клиента member_id = 10, которого еще нет в members:
❌ ОШИБКА СУБД: Execution failed
👉 Вывод: дочернюю таблицу нельзя импортировать раньше родительской!


---
## Тема 5. Вставка данных (INSERT) и перехват ошибок целостности

### 💡 Зачем это нужно:
1. Защита от SQL-инъекций (SQL Injection).
2. Проверка работы ограничений целостности.

### 📌 На доску:
```python
# ⛔ КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО:
cur.execute(f"INSERT INTO members VALUES ({id}, '{nick}')")

# ✅ ЕДИНСТВЕННО ПРАВИЛЬНЫЙ СПОСОБ (через плейсхолдеры ?):
cur.execute("INSERT INTO members (nickname, balance) VALUES (?, ?)", (nick, balance))
```

### 💻 Демонстрация: как база защищает данные


In [5]:
# Добавим корректного ПК и клиента с явным ID = 1
cur.execute("INSERT INTO hardware_pcs (pc_id, pc_number, zone_name, hourly_rate) VALUES (?, ?, ?, ?)",
            (1, 'PC-01', 'VIP', 400.0))
cur.execute("INSERT INTO members (member_id, nickname, balance) VALUES (?, ?, ?)",
            (1, 'CyberViper', 1200.0))
conn.commit()

# Попытка 1: Нарушить CHECK (отрицательный тариф)
try:
    cur.execute("INSERT INTO hardware_pcs (pc_number, zone_name, hourly_rate) VALUES (?, ?, ?)",
                ('PC-02', 'Standard', -50.0))
except sqlite3.IntegrityError as e:
    print(f"1. База отклонила отрицательный тариф: {e}")

# Попытка 2: Нарушить UNIQUE (дубликат ника)
try:
    cur.execute("INSERT INTO members (nickname, balance) VALUES (?, ?)",
                ('CyberViper', 500.0))
except sqlite3.IntegrityError as e:
    print(f"2. База отклонила дубликат ника: {e}")

# Попытка 3: Нарушить FOREIGN KEY (клиента с ID 999 не существует)
try:
    cur.execute("INSERT INTO game_sessions (member_id, pc_id, hours_played) VALUES (?, ?, ?)",
                (999, 1, 2.0))
except sqlite3.IntegrityError as e:
    print(f"3. База отклонила сессию с несуществующим клиентом: {e}")


1. База отклонила отрицательный тариф: CHECK constraint failed: hourly_rate > 0
2. База отклонила дубликат ника: UNIQUE constraint failed: members.nickname
3. База отклонила сессию с несуществующим клиентом: FOREIGN KEY constraint failed


---
## Тема 6. Модификация данных (UPDATE)

### 💡 Зачем это нужно:
Изменение статусов заказов, списание денег со счета, обновление тарифов.

### 📌 На доску:
```sql
UPDATE имя_таблицы
SET колонка_1 = новое_значение, колонка_2 = ...
WHERE условие_фильтрации;
```

### ⚠️ Главное правило безопасности UPDATE (записать на доску крупно):
> **UPDATE БЕЗ WHERE ИЗМЕНИТ ВСЕ СТРОКИ В ТАБЛИЦЕ!**  
> Если написать `UPDATE members SET balance = 0;` — все клиенты клуба мгновенно потеряют все деньги.

**Алгоритм безопасной работы:**
1. Сначала пишем `SELECT * FROM members WHERE member_id = 1;` и смотрим на строку.
2. Только затем пишем `UPDATE ... WHERE member_id = 1;`.
3. Обязательно вызываем `conn.commit()`, иначе изменения останутся в оперативной памяти и исчезнут.
4. Проверяем `cur.rowcount` — количество фактически обновленных строк.


In [6]:
# Демонстрация обновления баланса
print("Баланс ДО обновления:")
display(pd.read_sql_query("SELECT member_id, nickname, balance FROM members WHERE member_id = 1;", conn))

cur.execute("UPDATE members SET balance = balance + ? WHERE member_id = ?;", (500.0, 1))
conn.commit()

print(f"Обновлено строк: {cur.rowcount}")
print("Баланс ПОСЛЕ обновления:")
display(pd.read_sql_query("SELECT member_id, nickname, balance FROM members WHERE member_id = 1;", conn))


Баланс ДО обновления:


,member_id,nickname,balance
0,1,CyberViper,1200.0


Обновлено строк: 1
Баланс ПОСЛЕ обновления:


,member_id,nickname,balance
0,1,CyberViper,1700.0


---
## Тема 7. Удаление данных (DELETE) и каскадные правила

### 💡 Зачем это нужно:
Очистка тестовых записей, удаление отмененных броней.

### 📌 На доску:
```sql
DELETE FROM имя_таблицы WHERE условие_фильтрации;
```

### ⚙️ Поведение при удалении родителя:
1. **По умолчанию (RESTRICT / NO ACTION):** если у клиента есть сессии, удалить клиента нельзя — СУБД вернет ошибку `FOREIGN KEY constraint failed`.
2. **Каскадное удаление (`ON DELETE CASCADE`):** при удалении клиента СУБД **автоматически удаляет все его сессии**. Задается строго при создании таблицы в DDL!


In [7]:
# Добавим корректную сессию для клиента 1
cur.execute("INSERT INTO game_sessions (member_id, pc_id, hours_played) VALUES (?, ?, ?)",
            (1, 1, 3.0))
conn.commit()

# Попытка удалить клиента, у которого есть сессия (RESTRICT):
try:
    cur.execute("DELETE FROM members WHERE member_id = 1;")
except sqlite3.IntegrityError as e:
    print(f"Защита от удаления родителя сработала: {e}")


Защита от удаления родителя сработала: FOREIGN KEY constraint failed


---
## Тема 8. Управление транзакциями (COMMIT и ROLLBACK)

### 💡 Зачем это нужно (Аналогия из жизни):
Представьте банковский перевод или оплату игры:
1. Шаг 1: списать 300 рублей со счета клиента.
2. Шаг 2: продлить игровое время на компьютере.

Что будет, если после шага 1 выключится сервер или произойдет ошибка? Деньги списались, а время не добавилось!
**Транзакция гарантирует: либо выполняются ВСЕ действия цепочки, либо НЕ ВЫПОЛНЯЕТСЯ НИ ОДНО (принцип «Всё или ничего»).**

### 📌 На доску:
```python
try:
    cur.execute("UPDATE members SET balance = balance - 300 WHERE member_id = 1;")
    cur.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = 1;")
    conn.commit()    # ✅ Успех: все изменения зафиксированы на диск
    print("Транзакция успешно проведена.")
except Exception as e:
    conn.rollback()  # ❌ Ошибка: все незафиксированные действия отменены!
    print(f"Сбой транзакции, изменения откатаны: {e}")
```


In [8]:
# Демонстрация отката транзакции при ошибке
# У клиента баланс 1700. Попробуем списать 5000 (нарушит CHECK balance >= 0)
print("Состояние ДО аварийной транзакции:")
display(pd.read_sql_query("SELECT member_id, balance FROM members WHERE member_id = 1;", conn))

try:
    # 1. Меняем статус сессии
    cur.execute("UPDATE game_sessions SET status = 'completed' WHERE session_id = 1;")
    # 2. Списываем сумму больше депозита (вызовет IntegrityError из-за CHECK)
    cur.execute("UPDATE members SET balance = balance - 5000 WHERE member_id = 1;")
    conn.commit()
except sqlite3.IntegrityError as e:
    print(f"⚠️ Ошибка в цепочке действий ({e})! Выполняем ROLLBACK...")
    conn.rollback()

print("Состояние ПОСЛЕ ROLLBACK (статус сессии и баланс остались нетронутыми!):")
display(pd.read_sql_query("SELECT member_id, balance FROM members WHERE member_id = 1;", conn))
display(pd.read_sql_query("SELECT session_id, status FROM game_sessions WHERE session_id = 1;", conn))


Состояние ДО аварийной транзакции:


,member_id,balance
0,1,1700.0


⚠️ Ошибка в цепочке действий (CHECK constraint failed: balance >= 0)! Выполняем ROLLBACK...
Состояние ПОСЛЕ ROLLBACK (статус сессии и баланс остались нетронутыми!):


,member_id,balance
0,1,1700.0


,session_id,status
0,1,active


---
## Итоги лекции: Чек-лист для студентов перед лабораторной
1. Всегда включать `PRAGMA foreign_keys = ON;`.
2. Сначала проектировать схему DDL (`CREATE TABLE`), а не использовать `if_exists='replace'`.
3. Импортировать CSV через `if_exists='append'`, строго соблюдая очередность (родительские таблицы -> дочерние).
4. Любые переменные в SQL передавать через `?`.
5. Команды `UPDATE` и `DELETE` всегда писать с `WHERE`.
6. Связанные операции оборачивать в транзакцию с `conn.commit()` и `conn.rollback()`.
